<a href="https://colab.research.google.com/github/hubcborja/Trabajo_grado_ECIJG/blob/main/Variables_exogenas_dtt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Union variables exogenas y red de convenios de doble tributacion

In [24]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Importación de bases de datos

Desde la biblioteca del Gravity

Conte, M., P. Cotterlaz and T. Mayer (2022), "The CEPII Gravity database". CEPII Working Paper N°2022-05, July 2022.


In [16]:
gravity=pd.read_csv('/content/drive/MyDrive/Gravity/Gravity_V202211.csv', low_memory=False)
print(gravity.columns)
print(gravity['country_id_o'].count())
print(gravity.head())

Index(['year', 'country_id_o', 'country_id_d', 'iso3_o', 'iso3_d', 'iso3num_o',
       'iso3num_d', 'country_exists_o', 'country_exists_d',
       'gmt_offset_2020_o', 'gmt_offset_2020_d', 'distw_harmonic',
       'distw_arithmetic', 'distw_harmonic_jh', 'distw_arithmetic_jh', 'dist',
       'main_city_source_o', 'main_city_source_d', 'distcap', 'contig',
       'diplo_disagreement', 'scaled_sci_2021', 'comlang_off', 'comlang_ethno',
       'comcol', 'col45', 'legal_old_o', 'legal_old_d', 'legal_new_o',
       'legal_new_d', 'comleg_pretrans', 'comleg_posttrans',
       'transition_legalchange', 'comrelig', 'heg_o', 'heg_d', 'col_dep_ever',
       'col_dep', 'col_dep_end_year', 'col_dep_end_conflict', 'empire',
       'sibling_ever', 'sibling', 'sever_year', 'sib_conflict', 'pop_o',
       'pop_d', 'gdp_o', 'gdp_d', 'gdpcap_o', 'gdpcap_d', 'pop_source_o',
       'pop_source_d', 'gdp_source_o', 'gdp_source_d', 'gdp_ppp_o',
       'gdp_ppp_d', 'gdpcap_ppp_o', 'gdpcap_ppp_d', 'pop_pwt_o',

In [17]:
# Base de indices de los paises obtenida desde la biblioteca del Gravity
countries=pd.read_csv('/content/drive/MyDrive/Gravity/Countries_V202211.csv')
print(countries.columns)
countries.head()

Index(['country_id', 'iso3', 'iso3num', 'country', 'countrylong', 'first_year',
       'last_year', 'countrygroup_iso3', 'countrygroup_iso3num', 'iso2',
       'heg_iso3_2020', 'heg_iso3num_2020'],
      dtype='object')


,country_id,iso3,iso3num,country,countrylong,first_year,last_year,countrygroup_iso3,countrygroup_iso3num,iso2,heg_iso3_2020,heg_iso3num_2020
0,AFG,AFG,4.0,Afghanistan,Islamic Republic of Afghanistan,NaN,NaN,NaN,NaN,AF,NaN,NaN
1,ALB,ALB,8.0,Albania,Republic of Albania,NaN,NaN,NaN,NaN,AL,NaN,NaN
2,DZA,DZA,12.0,Algeria,People's Democratic Republic of Algeria,NaN,NaN,NaN,NaN,DZ,NaN,NaN
3,ASM,ASM,16.0,American Samoa,American Samoa,NaN,NaN,NaN,NaN,AS,USA,840.0
4,AND,AND,20.0,Andorra,Principality of Andorra,NaN,NaN,NaN,NaN,AD,NaN,NaN


In [18]:
base_convenios=pd.read_csv('https://raw.githubusercontent.com/hubcborja/Trabajo_grado_ECIJG/refs/heads/main/bases_datos/convenios_tributarios.csv')
# diccionario nombre de las columnas con artículo a nombre del artículo
renombrar_columnas = {
    'Source': 'indice_fuente_total',                   # Que tanto beneficia al pais fuente
    'PE': 'indice_establecimiento_permanente',         # Que tanto se le cobran a los establecimientos permanentes
    'WHT Rates': 'indice_tasas_retencion',             # Que tanto tienen de topes las tasas de retención
    'Other': 'indice_otros_derechos',                  # Indice de los otros derechos del convenio
    'UN': 'indice_modelo_onu',                          # Que tanto se parece al modelo ONU (que favorece a los estado en desarrollo)
    '5(3)(a)C': 'duracion_ep_construccion_meses',      # Construction PE length
    '5(3)(a)S': 'actividades_supervision_ep',          # Supervisory activities in PE
    '5(3)(b)': 'duracion_ep_servicios_meses',          # Service PE length
    '5(4)(a)': 'instalaciones_entrega_excluidas_ep',   # Delivery facilities excluded
    '5(4)(b)': 'inventario_entrega_excluido_ep',       # Delivery stock excluded
    '5(5)(b)': 'agente_con_inventario_ep',             # Agent maintaining stock included
    '5(6)': 'corredor_seguros_ep',                     # Insurance broker included
    '5(7)': 'extension_agente_dependiente_ep',         # Dependent agent extension
    '7(1)(b&c)': 'fuerza_atraccion_limitada',          # Limited force of attraction
    '7(3)': 'sin_deduccion_pagos_matriz',              # No deduction payments to head office
    '8(2)': 'impuesto_transporte_maritimo',            # Shared taxing right shipping
    '10(2)(a)Q': 'tasa_retencion_dividendos_calificados', # Qualifying dividend WHT rate
    '10(2)(a)T': 'umbral_participacion_dividendos',    # Threshold for qualified dividends
    '10(2)(b)': 'tasa_retencion_dividendos_portafolio',# Portfolio dividend WHT rate
    '11(2)': 'tasa_retencion_intereses',               # Interest WHT rate
    '11(2)F': 'tasa_retencion_intereses_financieras',  # Financial inst. interest rate
    '12(2)': 'tasa_retencion_regalias',                # Royalties WHT rate
    '12(2)C': 'tasa_retencion_regalias_derechos_autor',# Copyright royalties rate
    '12(2)E': 'tasa_retencion_regalias_equipos',       # Equipment royalties rate
    '12(A)': 'tasa_retencion_servicios_tecnicos',      # Technical service fees rate
    '13(4)': 'ganancia_capital_inmobiliaria',          # Capital gains (land rich)
    '13(5)': 'ganancia_capital_otras_acciones',        # Capital gains (other shares)
    '14': 'servicios_personales_independientes',       # Independent personal services
    '16(2)': 'oficiales_alta_gerencia',                # Top-level managerial officials
    '21(3)': 'impuesto_fuente_otros_ingresos',         # Source taxation other income
    '25B(5)': 'arbitraje_obligatorio_vinculante',      # Mandatory binding arbitration
    '27': 'asistencia_cobro_impuestos',                # Assistance in tax collection
    '29': 'regla_general_anti_abuso'                   # General anti-abuse rule (GAAR)
}


base_convenios.rename(columns=renombrar_columnas, inplace=True)
base_convenios.head()

,Country A,Country B,Type,Date of signature,Effective,Status,indice_fuente_total,WHTrates,indice_establecimiento_permanente,indice_otros_derechos,...,tasa_retencion_regalias_equipos,tasa_retencion_servicios_tecnicos,ganancia_capital_inmobiliaria,ganancia_capital_otras_acciones,servicios_personales_independientes,oficiales_alta_gerencia,impuesto_fuente_otros_ingresos,arbitraje_obligatorio_vinculante,asistencia_cobro_impuestos,regla_general_anti_abuso
0,Albania,Israel,Original,2021,2022.0,In Force,0.30,0.31,0.45,0.13,...,5,0,YES,NO,NO,NO,NO,NO,NO,PPT
1,Albania,Turkey,Original,1994,1997.0,In Force,0.19,0.38,0.06,0.13,...,10,0,NO,NO,YES,NO,NO,NO,NO,NaN
2,Albania,Romania,Original,1994,1995.0,In Force,0.47,0.47,0.58,0.38,...,15,0,NO,NO,YES,NO,YES,NO,NO,NaN
3,Albania,Italy,Original,1994,2000.0,In Force,0.19,0.25,0.19,0.13,...,5,0,NO,NO,YES,NO,NO,NO,NO,NaN
4,Albania,Russia,Original,1995,1998.0,In Force,0.23,0.38,0.06,0.25,...,10,0,NO,NO,YES,NO,YES,NO,NO,NaN


In [19]:
base_convenios[base_convenios['Country A']=='Colombia']

,Country A,Country B,Type,Date of signature,Effective,Status,indice_fuente_total,WHTrates,indice_establecimiento_permanente,indice_otros_derechos,...,tasa_retencion_regalias_equipos,tasa_retencion_servicios_tecnicos,ganancia_capital_inmobiliaria,ganancia_capital_otras_acciones,servicios_personales_independientes,oficiales_alta_gerencia,impuesto_fuente_otros_ingresos,arbitraje_obligatorio_vinculante,asistencia_cobro_impuestos,regla_general_anti_abuso
647,Colombia,Spain,Original,2005,2008.0,In Force,0.23,0.34,0.22,0.13,...,10,10,YES,NO,NO,NO,NO,NO,YES,NaN
648,Colombia,Mexico,Original,2009,2013.0,In Force,0.44,0.34,0.47,0.50,...,10,10,YES,YES,NO,NO,YES,NO,YES,LOB
649,Colombia,South Korea,Original,2010,2015.0,In Force,0.40,0.47,0.47,0.25,...,10,10,YES,YES,NO,NO,NO,NO,YES,PARTIAL
650,Colombia,Portugal,Original,2010,2016.0,In Force,0.53,0.50,0.59,0.50,...,10,10,YES,YES,YES,NO,YES,NO,YES,PPT
651,Colombia,India,Original,2011,2014.0,In Force,0.64,0.44,0.97,0.50,...,10,10,YES,YES,YES,NO,YES,NO,YES,PPT
652,Colombia,Czechia,Original,2012,2016.0,In Force,0.55,0.44,0.97,0.25,...,10,10,NO,YES,NO,NO,YES,NO,NO,PPT
653,Colombia,France,Original,2015,2023.0,In Force,0.30,0.31,0.34,0.25,...,10,0,YES,YES,NO,NO,NO,NO,YES,PPT
654,Colombia,United Kingdom,Original,2016,2020.0,In Force,0.28,0.31,0.09,0.43,...,10,0,YES,YES,NO,NO,YES,NO,YES,PPT
655,Colombia,Italy,Original,2018,2022.0,In Force,0.45,0.31,0.47,0.57,...,10,0,YES,YES,YES,NO,YES,NO,YES,PPT
656,Colombia,Japan,Original,2018,2023.0,In Force,0.38,0.25,0.47,0.43,...,2,0,YES,YES,NO,NO,YES,NO,YES,PARTIAL


## Conteos de Configuraciones

In [23]:
# Colocar la base en networkx y calcular el grado (nodos o paises que tiene convenio)
base_convenios_23=base_convenios[base_convenios['Date of signature']<=2023]
grafo=nx.from_pandas_edgelist(base_convenios_23,
                            source='Country A',
                            target='Country B',
                            edge_attr='Date of signature')

Grados_paises = dict(grafo.degree())
Grados_paises

{'Albania': 43,
 'Israel': 17,
 'Turkey': 39,
 'Romania': 37,
 'Italy': 46,
 'Russia': 33,
 'Czechia': 40,
 'North Macedonia': 50,
 'Sweden': 35,
 'Norway': 37,
 'Moldova': 54,
 'Montenegro': 12,
 'South Korea': 40,
 'Kuwait': 33,
 'Estonia': 16,
 'Germany': 44,
 'Spain': 37,
 'Saudi Arabia': 26,
 'Kosovo': 21,
 'Switzerland': 41,
 'Belgium': 42,
 'Bosnia and Herzegovina': 36,
 'Bulgaria': 25,
 'China': 107,
 'Croatia': 21,
 'Egypt': 59,
 'France': 63,
 'Greece': 15,
 'Iceland': 7,
 'India': 100,
 'Ireland': 21,
 'Latvia': 17,
 'Malaysia': 37,
 'Malta': 22,
 'Netherlands': 36,
 'Poland': 32,
 'Qatar': 34,
 'Serbia': 20,
 'Singapore': 35,
 'Slovenia': 15,
 'United Arab Emirates': 53,
 'United Kingdom': 58,
 'Hungary': 28,
 'Austria': 31,
 'Algeria': 36,
 'Tunisia': 55,
 'Libya': 13,
 'Morocco': 66,
 'Indonesia': 70,
 'Syria': 37,
 'Jordan': 38,
 'South Africa': 79,
 'Canada': 44,
 'Oman': 19,
 'Bahrain': 21,
 'Lebanon': 29,
 'Ukraine': 73,
 'Portugal': 26,
 'Iran': 50,
 'Mauritania': 5,

In [29]:
# Estrellas

def contar_estrellas_de_k_radios(Grafo_G, grados_dict, k=4):
  estrellas_totales=0
  for nodo, grado in grados_dict.items():
    if grado<=k:
      continue
    estrellas_nodo=math.comb(grado, k)
    estrellas_totales += estrellas_nodo

  return print(f'Hay',estrellas_totales,'estrellas en el grafo')

contar_estrellas_de_k_radios(grafo, Grados_paises)

Hay 23780640 estrellas en el grafo


In [34]:
# Tripletes y Triángulos

triangulos=nx.triangles(grafo)
print('total de triangulos por pais:',triangulos)

print('total de triangulos:', sum(nx.triangles(grafo).values()) // 3)

total de triangulos por pais: {'Albania': 212, 'Israel': 68, 'Turkey': 230, 'Romania': 232, 'Italy': 250, 'Russia': 151, 'Czechia': 241, 'North Macedonia': 360, 'Sweden': 154, 'Norway': 148, 'Moldova': 469, 'Montenegro': 38, 'South Korea': 209, 'Kuwait': 205, 'Estonia': 65, 'Germany': 235, 'Spain': 177, 'Saudi Arabia': 121, 'Kosovo': 35, 'Switzerland': 210, 'Belgium': 215, 'Bosnia and Herzegovina': 208, 'Bulgaria': 143, 'China': 1485, 'Croatia': 99, 'Egypt': 563, 'France': 294, 'Greece': 55, 'Iceland': 14, 'India': 1271, 'Ireland': 87, 'Latvia': 71, 'Malaysia': 211, 'Malta': 99, 'Netherlands': 179, 'Poland': 203, 'Qatar': 174, 'Serbia': 96, 'Singapore': 177, 'Slovenia': 59, 'United Arab Emirates': 269, 'United Kingdom': 254, 'Hungary': 153, 'Austria': 182, 'Algeria': 261, 'Tunisia': 436, 'Libya': 49, 'Morocco': 534, 'Indonesia': 833, 'Syria': 310, 'Jordan': 326, 'South Africa': 651, 'Canada': 185, 'Oman': 82, 'Bahrain': 81, 'Lebanon': 191, 'Ukraine': 975, 'Portugal': 75, 'Iran': 548, '

In [31]:
# Cliques tetaedros

cliques=nx.find_cliques(grafo)

total_cliques=0
for clique in cliques:
  if len(clique)>=4:
    total_cliques+=1

print('total de cliques de 4 puntos', total_cliques)

total de cliques de 4 puntos 2603


## Union de bases de datos y manejo a pyTorch

In [35]:
# Seleccionar solo las columnas de traducción de la base 'countries'
df_country_map = countries[['country', 'iso3']].copy()

# Limpiar y normalizar los nombres si es necesario (ej. eliminar espacios extra)
df_country_map['country'] = df_country_map['country'].str.strip()

# Mapeo para pais A
base_convenios['iso3_A'] = base_convenios['Country A'].map(
    df_country_map.set_index('country')['iso3']
)

# Mapeo para pais B
base_convenios['iso3_B'] = base_convenios['Country B'].map(
    df_country_map.set_index('country')['iso3']
)

# Convertir la fecha de firma a año
base_convenios['year_signed'] = pd.to_datetime(
    base_convenios['Date of signature'], errors='coerce'
).dt.year

# Mantener solo las columnas necesarias
base_convenios = base_convenios[['iso3_A', 'iso3_B', 'year_signed', 'Status', 'Effective', 'Type']]

print('Conteo antes de eliminar filas que no tuvieron ISO3: ', base_convenios['iso3_A'].count())
# Eliminar filas donde no se pudo encontrar el ISO3
base_convenios.dropna(subset=['iso3_A', 'iso3_B'], inplace=True)
print('Conteo después de eliminar filas que no tuvieron ISO3:: ', base_convenios['iso3_A'].count())

base_convenios.head(1)

Conteo antes de eliminar filas que no tuvieron ISO3:  1927
Conteo después de eliminar filas que no tuvieron ISO3::  1878


,iso3_A,iso3_B,year_signed,Status,Effective,Type
0,ALB,ISR,1970,In Force,2022.0,Original


In [36]:
# Columnas par para unión de las bases
gravity['iso3_pair'] = gravity.apply(
    lambda row: tuple(sorted((row['iso3_o'], row['iso3_d']))), axis=1
)

base_convenios['iso3_pair'] = base_convenios.apply(
    lambda row: tuple(sorted((row['iso3_A'], row['iso3_B']))), axis=1
)

df_treaty_signed = base_convenios.groupby('iso3_pair')['year_signed'].min().reset_index()
df_treaty_signed.rename(columns={'year_signed': 'year_first_signed'}, inplace=True)

# Merge de gravity y la tabla de firma
gravity_convenios = pd.merge(
    gravity,
    df_treaty_signed,
    on='iso3_pair',
    how='left'
)

# Variable binaria de CDT vigente
gravity_convenios['tiene_CDT_vigente'] = np.where(
    gravity_convenios['year'] >= gravity_convenios['year_first_signed'],
    1,
    0
)

# Tratar pares que nunca firmaron un CDT
gravity_convenios['tiene_CDT_vigente'] = gravity_convenios['tiene_CDT_vigente'].fillna(0)

In [37]:
gravity_convenios[gravity_convenios['country_id_o']=='COL']

,year,country_id_o,country_id_d,iso3_o,iso3_d,iso3num_o,iso3num_d,country_exists_o,country_exists_d,gmt_offset_2020_o,...,entry_tp_d,tradeflow_comtrade_o,tradeflow_comtrade_d,tradeflow_baci,manuf_tradeflow_baci,tradeflow_imf_o,tradeflow_imf_d,iso3_pair,year_first_signed,tiene_CDT_vigente
857808,1948,COL,ABW,COL,ABW,170.0,533.0,1,0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(ABW, COL)",NaN,0
857809,1949,COL,ABW,COL,ABW,170.0,533.0,1,0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(ABW, COL)",NaN,0
857810,1950,COL,ABW,COL,ABW,170.0,533.0,1,0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(ABW, COL)",NaN,0
857811,1951,COL,ABW,COL,ABW,170.0,533.0,1,0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(ABW, COL)",NaN,0
857812,1952,COL,ABW,COL,ABW,170.0,533.0,1,0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(ABW, COL)",NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
876451,2017,COL,ZWE,COL,ZWE,170.0,716.0,1,1,-5.0,...,70.0,4.025,4.600,6.986,6.986,4.026,NaN,"(COL, ZWE)",NaN,0
876452,2018,COL,ZWE,COL,ZWE,170.0,716.0,1,1,-5.0,...,41.0,117.353,51.607,166.336,166.326,117.354,0.023,"(COL, ZWE)",NaN,0
876453,2019,COL,ZWE,COL,ZWE,170.0,716.0,1,1,-5.0,...,36.0,12.131,12.078,14.153,14.153,12.132,0.002,"(COL, ZWE)",NaN,0
876454,2020,COL,ZWE,COL,ZWE,170.0,716.0,1,1,-5.0,...,NaN,5.404,8.335,8.997,8.997,5.405,0.001,"(COL, ZWE)",NaN,0


In [38]:
# Extraer los códigos ISO3 reales de las columnas de origen y destino
nodos_origen = set(gravity_convenios['iso3_o'].dropna().unique())
nodos_destino = set(gravity_convenios['iso3_d'].dropna().unique())

# Unir ambos sets para tener todos los nodos únicos y ordenarlos
nodos_unicos = sorted(list(nodos_origen | nodos_destino))
num_nodos = len(nodos_unicos)

print(f"Número total de nodos (países) reales en la red: {num_nodos}")

# Crear los diccionarios de mapeo bidireccional
node_to_index = {iso: idx for idx, iso in enumerate(nodos_unicos)}
index_to_node = {idx: iso for idx, iso in enumerate(nodos_unicos)}

id_colombia = node_to_index.get('COL')
print(f"El índice numérico para Colombia (COL) será: {id_colombia}")

# Aplicar el mapeo para crear las columnas de índices en TensorFlow
gravity_convenios['node_idx_o'] = gravity_convenios['iso3_o'].map(node_to_index)
gravity_convenios['node_idx_d'] = gravity_convenios['iso3_d'].map(node_to_index)

Número total de nodos (países) reales en la red: 243
El índice numérico para Colombia (COL) será: 45


In [40]:
# Mapear el origen y el destino a sus respectivos índices en la base principal
gravity_convenios['node_idx_o'] = gravity_convenios['iso3_o'].map(node_to_index)
gravity_convenios['node_idx_d'] = gravity_convenios['iso3_d'].map(node_to_index)

print('paises_nulos_origen: ', gravity_convenios['node_idx_o'].isnull().sum())
print('paises_nulos_destino: ', gravity_convenios['node_idx_d'].isnull().sum())


print(gravity_convenios.head())

paises_nulos_origen:  0
paises_nulos_destino:  0
   year country_id_o country_id_d iso3_o iso3_d  iso3num_o  iso3num_d  \
0  1948          ABW          ABW    ABW    ABW      533.0      533.0   
1  1949          ABW          ABW    ABW    ABW      533.0      533.0   
2  1950          ABW          ABW    ABW    ABW      533.0      533.0   
3  1951          ABW          ABW    ABW    ABW      533.0      533.0   
4  1952          ABW          ABW    ABW    ABW      533.0      533.0   

   country_exists_o  country_exists_d  gmt_offset_2020_o  ...  \
0                 0                 0                NaN  ...   
1                 0                 0                NaN  ...   
2                 0                 0                NaN  ...   
3                 0                 0                NaN  ...   
4                 0                 0                NaN  ...   

   tradeflow_comtrade_d  tradeflow_baci  manuf_tradeflow_baci  \
0                   NaN             NaN                 

In [41]:
# Definir las variables que son estrictamente de NODO
# Tomamos las variables con sufijo '_o' (origen) para representar al país
features_nodos = ['gdp_o', 'pop_o', 'gdpcap_o'] # Agrega aquí más variables macroeconómicas si las tienes

# Extraer un DataFrame a nivel de nodo (eliminando duplicados de la estructura origen-destino)
columnas_extraer = ['year', 'node_idx_o'] + features_nodos
df_nodos = gravity_convenios[columnas_extraer].dropna(subset=['node_idx_o']).drop_duplicates(subset=['year', 'node_idx_o']).copy()

# Convertir el índice a entero para usarlo en matrices
df_nodos['node_idx_o'] = df_nodos['node_idx_o'].astype(int)

# Definir dimensiones
# Filtrar años si es necesario (ej. desde 2000 en adelante). Aquí tomamos todos los disponibles.
anios = sorted(df_nodos['year'].unique())
T = len(anios)
F = len(features_nodos)

print(f"Dimensiones esperadas -> Tiempo (T): {T}, Nodos (N): {num_nodos}, Variables (F): {F}")

X_t = [] # Lista para almacenar las matrices anuales

# Imputador y escalador
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_t = [] # Lista para almacenar las matrices anuales

# Construir el tensor año por año
for t in anios:
    df_t = df_nodos[df_nodos['year'] == t]

    # Matriz vacía [N, F]
    matriz_t = np.full((num_nodos, F), np.nan)

    if not df_t.empty:
        indices = df_t['node_idx_o'].values
        valores = df_t[features_nodos].values
        matriz_t[indices] = valores

    for col in range(F):
        if np.all(np.isnan(matriz_t[:, col])):
            matriz_t[:, col] = 0.0

    # Ahora sí imputamos los NaNs de los países faltantes (si los hay)
    matriz_t_imputada = imputer.fit_transform(matriz_t)

    # Normalizar
    matriz_t_escalada = scaler.fit_transform(matriz_t_imputada)

    X_t.append(matriz_t_escalada)

# Convertir a Tensor de NumPy
X_tensor_np = np.array(X_t)
print(f"Forma final del tensor de características (X): {X_tensor_np.shape}")

Dimensiones esperadas -> Tiempo (T): 74, Nodos (N): 243, Variables (F): 3
Forma final del tensor de características (X): (74, 243, 3)


### Matrices de Adyacencia Temporales

In [42]:
A_t = []

for t in anios:
    # Crear una matriz vacía de ceros con dimensiones [N, N] para el año t
    matriz_A = np.zeros((num_nodos, num_nodos), dtype=np.float32)

    # Filtrar solo los registros de ese año donde hay un tratado vigente
    df_t = gravity_convenios[(gravity_convenios['year'] == t) & (gravity_convenios['tiene_CDT_vigente'] == 1)].dropna(subset=['node_idx_o', 'node_idx_d'])

    if not df_t.empty:
        # Extraer los índices de origen y destino como enteros
        idx_origen = df_t['node_idx_o'].astype(int).values
        idx_destino = df_t['node_idx_d'].astype(int).values

        # Llenar la matriz con 1s en las coordenadas correspondientes
        matriz_A[idx_origen, idx_destino] = 1.0

        # Hacer la matriz simétrica (los grafos de convenios son no dirigidos)
        matriz_A[idx_destino, idx_origen] = 1.0

    A_t.append(matriz_A)

# Convertir la lista de matrices a un tensor 3D de NumPy
A_tensor_np = np.array(A_t)
print(f"Forma final del tensor de adyacencia (A): {A_tensor_np.shape}")

# Verificación rápida de la densidad de la red en el primer y último año
print(f"Enlaces en el primer año ({anios[0]}): {int(np.sum(A_tensor_np[0]) / 2)}")
print(f"Enlaces en el último año ({anios[-1]}): {int(np.sum(A_tensor_np[-1]) / 2)}")

Forma final del tensor de adyacencia (A): (74, 243, 243)
Enlaces en el primer año (1948): 0
Enlaces en el último año (2021): 1876


### Convertir a tensor denso de TensorFlow

In [43]:
# Función para convertir np a tensores
X_tf = tf.convert_to_tensor(X_tensor_np, dtype=tf.float32)

print(f"Tensor X en TensorFlow: {X_tf.shape}")
print(f"Tipo de dato de X: {X_tf.dtype}\n")

# Función para convertir matrices densas a tensores dispersos (Sparse Tensors)
# Esto es obligatorio para que funcione con tu capa GraphConvolution
def numpy_a_sparse_tensor(matriz_np):
    """
    Convierte una matriz NumPy 2D a un tf.SparseTensor de TensorFlow.
    Extrae solo las coordenadas donde hay un enlace (1.0).
    """
    # Encontrar las coordenadas (fila, columna) de los enlaces
    indices = np.argwhere(matriz_np != 0)

    # Extraer los valores en esas coordenadas
    valores = matriz_np[indices[:, 0], indices[:, 1]]

    # Crear el objeto SparseTensor
    tensor_disperso = tf.SparseTensor(
        indices=indices,
        values=tf.cast(valores, tf.float32),
        dense_shape=matriz_np.shape
    )

    # Reordenar los índices (TensorFlow lo exige para multiplicaciones eficientes)
    return tf.sparse.reorder(tensor_disperso)

# Crear una lista de tensores dispersos para la adyacencia (uno por año)
A_tf_sparse_list = []

for i in range(T):
    A_sparse_t = numpy_a_sparse_tensor(A_tensor_np[i])
    A_tf_sparse_list.append(A_sparse_t)

# Verificamos el tensor del último año (2021)
A_ejemplo = A_tf_sparse_list[-1]
print(f"Adyacencia del último año convertida a SparseTensor.")
print(f"Forma original del grafo en ese año: {A_ejemplo.dense_shape}")
print(f"Número de valores almacenados en memoria (aristas x 2): {len(A_ejemplo.values)}")

Tensor X en TensorFlow: (74, 243, 3)
Tipo de dato de X: <dtype: 'float32'>

Adyacencia del último año convertida a SparseTensor.
Forma original del grafo en ese año: [243 243]
Número de valores almacenados en memoria (aristas x 2): 3752


# Creación del Dyn-IVGAE-GA